<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/Sistema-charge-gridd/blob/main/entregas/ChargeGrid_Intelligence_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq
print("✅ Cliente Groq pronto — a IA roda na nuvem (Groq), sem precisar instalar nem baixar modelo nenhum aqui.")

In [ ]:
import os

# Baixa os arquivos direto do GitHub — não precisa mais fazer upload manual
# toda vez. ev_chargegrid.py agora fala com o banco de verdade (Postgres,
# Supabase), então os dados aqui sempre vêm atualizados, sem precisar
# reenviar nada depois de testar o totem/dashboard.
!curl -fsSL -o ev_chargegrid.py https://raw.githubusercontent.com/pedrosampaiom2007-gif/Sistema-charge-gridd/main/entregas/ev_chargegrid.py
!curl -fsSL -o dados_rag.json https://raw.githubusercontent.com/pedrosampaiom2007-gif/Sistema-charge-gridd/main/entregas/dados_rag.json
!curl -fsSL -o modelo_demanda.pkl https://raw.githubusercontent.com/pedrosampaiom2007-gif/Sistema-charge-gridd/main/entregas/modelo_demanda.pkl
!pip install -q psycopg2-binary python-dotenv

# Credenciais: ficam só no "Secrets" do Colab (ícone de chave 🔑 na barra
# lateral esquerda), nunca aparecem no notebook nem no GitHub.
# Secrets necessários: DATABASE_URL (banco) e GROQ_API_KEY (IA).
from google.colab import userdata
os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

arquivos_obrigatorios = ['ev_chargegrid.py', 'dados_rag.json', 'modelo_demanda.pkl']
sucesso = all(arquivo in os.listdir('.') for arquivo in arquivos_obrigatorios)

if sucesso:
    print("\n✅ Tudo pronto! Arquivos baixados do GitHub e credenciais carregadas. Pode prosseguir.")
else:
    print("❌ ERRO: algum arquivo não foi baixado corretamente. Rode a célula de novo.")

In [ ]:
import os
import json
import unicodedata
from groq import Groq

# Importa as 4 funções de leitura do motor do Raul
from ev_chargegrid import (
    listar_sessoes_ativas,
    obter_status_estacoes,
    obter_faturamento_dia,
    contar_sessoes_dia,
    inicializar_banco
)

inicializar_banco()  # garante que o banco existe antes de qualquer leitura

# Carrega os dados históricos reais do Kevin
# Substitui os 12 documentos fixos e inventados do Sprint 2
with open('dados_rag.json', 'r', encoding='utf-8') as f:
    dados_rag = json.load(f)

documentos = dados_rag['frases_contexto_rag']

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODELO = "llama-3.1-8b-instant"

SYSTEM_PROMPT = """
[1] IDENTIDADE:
Você é o assistente inteligente do Charge Grid Intelligence, um sistema de gestão de
eletropostos para operações comerciais no contexto do EV Challenge 2026.

[2] CONTEXTO:
O Charge Grid Intelligence é um sistema voltado para postos comerciais e operadores
de frotas que precisam gerenciar eletropostos de alto fluxo de forma eficiente.
Você ajuda de duas formas: (a) consultando dados reais do sistema — sessões de
recarga, receita por ponto de carga, disponibilidade dos carregadores — e (b)
funcionando como um guia de bolso pro motorista, tirando dúvidas gerais sobre
carros elétricos (autonomia, tipos de conector, cuidados com a bateria, como
funciona a recarga).
A partir do Sprint 3, o chatbot tem acesso a dois tipos de dados sobre o sistema:
- DADOS EM TEMPO REAL: estado atual do sistema (carregadores ativos, faturamento de hoje)
- DADOS HISTÓRICOS: análise de 60 sessões reais da base SP2 (receita por carregador,
  pico de demanda, ticket médio, eficiência do DLB)

[3] REGRAS:
- Responda sobre o sistema Charge Grid Intelligence, operação de eletropostos, e
  dúvidas gerais de motoristas sobre carros elétricos.
- Se a pergunta não tiver relação nenhuma com recarga, carros elétricos ou o
  sistema, diga: "Só consigo ajudar com questões relacionadas a carros
  elétricos e ao Charge Grid Intelligence."
- Nunca invente dados do sistema (valores de consumo, faturamento) nem
  especificações técnicas exatas de um modelo específico de carro — se não
  tiver certeza sobre um modelo específico, diga isso claramente em vez de
  arriscar um número.
- Não opine sobre qual marca de carro ou rede de recarga é "melhor" — explique
  conceitos, não compare produtos.
- "Gasto pessoal do motorista logado" e "faturamento/receita total do sistema"
  são coisas DIFERENTES — nunca confunda os dois. Gasto pessoal é o que aquele
  motorista específico pagou; faturamento total é dado de negócio, somando
  todos os clientes. Se a pergunta for "quanto eu gastei" ou parecida, use
  APENAS o dado de "gasto pessoal do motorista logado" quando ele estiver no
  contexto — nunca responda com o faturamento total do sistema nesse caso.
- Se perguntarem sobre faturamento, receita, ticket médio ou qualquer outro
  número de negócio do sistema e esse dado NÃO estiver no contexto fornecido,
  não invente nem estime — diga que essa informação é restrita à gestão e
  não está disponível por aqui.
- Quando tiver dados em tempo real disponíveis no contexto, priorize-os sobre o histórico.

[4] TOM DE VOZ:
Seja claro, objetivo e use linguagem acessível, sem jargões técnicos
desnecessários. Responda sempre em português brasileiro.

[5] CONTEXTO DO SISTEMA:
- O sistema atende postos comerciais e frotas com múltiplos pontos de carga e alta rotatividade
- A cobrança é feita por kWh consumido com tarifa dinâmica por horário e ocupação
- O chatbot orienta gestores e operadores sobre consumo, faturamento e disponibilidade do sistema
- Picos de demanda são previstos e tarifados para evitar sobrecarga na infraestrutura elétrica
"""

# ─── Palavras-chave que ativam o roteador de tempo real ──────────────────────
# Perguntas com essas palavras consultam o banco ao vivo (Postgres/Supabase).
# CORREÇÃO: lista expandida — palavras curtas como "pico", "kwh", "dlb"
# antes eram descartadas pelo filtro len(p) > 3. Agora o roteador é separado
# da busca textual, então esse problema não existe mais.
PALAVRAS_TEMPO_REAL = [
    "agora", "hoje", "atual", "ativo", "ativa", "livre", "ocupado", "ocupada",
    "faturamento", "sessões de hoje", "quantas sessões", "status",
    "disponível", "carregando"
]

# ─── RAG: busca por relevância ───────────────────────────────────────────────
_STOPWORDS = {
    "a", "as", "ao", "aos", "com", "como", "da", "das", "de", "do", "dos", "e",
    "ele", "ela", "em", "essa", "esse", "esta", "este", "eu", "foi", "ha",
    "isso", "ja", "la", "mais", "mas", "me", "meu", "minha", "muito", "na",
    "nao", "nas", "no", "nos", "num", "o", "os", "ou", "para", "pela", "pelo",
    "por", "pra", "pro", "qual", "quais", "quando", "quanto", "quantos", "que",
    "se", "sem", "ser", "sao", "so", "sua", "seu", "tem", "ter", "um", "uma",
    "voce", "vc",
}

def _normalizar(texto: str) -> str:
    sem_acento = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in sem_acento if not unicodedata.combining(c))

def _tokenizar(texto: str) -> list:
    palavra, palavras = [], []
    for c in _normalizar(texto):
        if c.isalnum():
            palavra.append(c)
        elif palavra:
            palavras.append("".join(palavra)); palavra = []
    if palavra:
        palavras.append("".join(palavra))
    return palavras

def _casa(termo: str, palavras_doc: set) -> bool:
    if termo in palavras_doc:
        return True
    return len(termo) >= 5 and any(p.startswith(termo) for p in palavras_doc)

def buscar_documentos(pergunta: str, limite: int = 5) -> list:
    """Documentos ordenados por quantos termos da pergunta eles contêm."""
    termos = [p for p in _tokenizar(pergunta) if len(p) >= 3 and p not in _STOPWORDS]
    if not termos:
        return []
    pontuados = []
    for doc in documentos:
        palavras_doc = set(_tokenizar(doc))
        pontos = sum(1 for t in termos if _casa(t, palavras_doc))
        if pontos:
            pontuados.append((pontos, doc))
    pontuados.sort(key=lambda par: par[0], reverse=True)
    return [doc for _, doc in pontuados[:limite]]


# ─── RAG: buscar contexto com roteador inteligente ───────────────────────────
# CORREÇÃO: o RAG histórico agora só roda quando a pergunta NÃO é de tempo real.
# Antes, os dois blocos sempre rodavam juntos — isso gerava contexto redundante
# e confuso para o modelo (dados do banco misturados com dados da planilha
# para a mesma pergunta).
# Agora: pergunta de tempo real → só banco. Pergunta histórica → só planilha.
#
# Este notebook mantém acesso total (equivalente ao acesso_gestao=True de
# entregas/chatbot.py) — é a ferramenta de uso interno da equipe, não a que o
# motorista usa no app. Lá o padrão é o contrário: sem token de admin, o chat
# não enxerga faturamento nem histórico comercial.
def buscar_contexto(pergunta: str) -> str:
    pergunta_lower = pergunta.lower()
    usa_tempo_real = any(p in pergunta_lower for p in PALAVRAS_TEMPO_REAL)

    partes = []

    if usa_tempo_real:
        # Busca dados ao vivo no banco Postgres via funções do Raul
        try:
            sessoes_ativas   = listar_sessoes_ativas()
            status_estacoes  = obter_status_estacoes()
            faturamento_hoje = obter_faturamento_dia()
            sessoes_hoje     = contar_sessoes_dia()

            livres   = [k for k, v in status_estacoes.items() if v == 'Livre']
            ocupadas = [k for k, v in status_estacoes.items() if v == 'Ocupada']

            partes.append("[DADOS EM TEMPO REAL — banco Postgres]")
            partes.append(f"Estações ocupadas agora: {ocupadas if ocupadas else 'nenhuma'}")
            partes.append(f"Estações livres agora: {livres}")
            partes.append(f"Faturamento de hoje (sessões pagas): R$ {faturamento_hoje:.2f}")
            partes.append(f"Total de sessões iniciadas hoje: {sessoes_hoje}")

            for s in sessoes_ativas:
                partes.append(
                    f"Sessão ativa — Estação {s['estacao']}: usuário {s['usuario']}, "
                    f"{s['kwh']:.2f} kWh consumidos, valor acumulado R$ {s['valor']:.2f}, "
                    f"pagamento via {s['pagamento']}."
                )
        except Exception as e:
            partes.append(f"[AVISO] Banco indisponível: {e}")

    else:
        # RAG histórico: busca nos dados reais da planilha SP2.
        # CORREÇÃO: a busca casava QUALQUER palavra da pergunta dentro do
        # texto do documento — inclusive "o", "de", "e" — então quase toda
        # pergunta trazia os mesmos 5 documentos de receita, mesmo quando o
        # assunto era bateria de carro elétrico. Agora: ignora palavra vazia,
        # ignora acento, compara palavra inteira e ordena pelo número de
        # termos que casaram. Mesma lógica de entregas/chatbot.py.
        relevantes = buscar_documentos(pergunta)
        if relevantes:
            partes.append("[DADOS HISTÓRICOS — planilha SP2, 60 sessões reais]")
            partes.extend(relevantes[:5])

    return "\n".join(partes)

# ─── Histórico ────────────────────────────────────────────────────────────────
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

def chat(pergunta: str) -> str:
    contexto = buscar_contexto(pergunta)
    if contexto:
        mensagem = f"Contexto do sistema:\n{contexto}\n\nPergunta: {pergunta}"
    else:
        mensagem = pergunta
    historico.append({"role": "user", "content": mensagem})
    resposta = client.chat.completions.create(model=MODELO, messages=historico)
    conteudo = resposta.choices[0].message.content
    historico.append({"role": "assistant", "content": conteudo})
    return conteudo

print(f"✅ Chatbot Sprint 3 pronto! RAG com {len(documentos)} fragmentos históricos reais indexados.")
print(f"   Roteador de tempo real ativo — {len(PALAVRAS_TEMPO_REAL)} palavras-chave monitoradas.")

In [ ]:

import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("<h3>🔋 Charge Grid Intelligence — CGI Assistant</h3>"))

saida = widgets.Output(layout=widgets.Layout(
    border="1px solid #ccc", min_height="200px", padding="10px"
))
campo = widgets.Text(
    placeholder="Digite sua pergunta...",
    layout=widgets.Layout(width="75%")
)
botao = widgets.Button(
    description="Enviar",
    button_style="primary",
    layout=widgets.Layout(width="20%")
)
btn_limpar = widgets.Button(description="Limpar", layout=widgets.Layout(width="10%"))

def ao_enviar(b):
    pergunta = campo.value.strip()
    if not pergunta:
        return
    campo.value = ""
    with saida:
        print(f"👤 Você: {pergunta}")
        print(f"🤖 Bot: {chat(pergunta)}")
        print("-" * 50)

def ao_limpar(b):
    saida.clear_output()
    historico.clear()
    historico.append({"role": "system", "content": SYSTEM_PROMPT})

botao.on_click(ao_enviar)
btn_limpar.on_click(ao_limpar)
campo.on_submit(ao_enviar)  # Enter também envia

display(widgets.HBox([campo, botao, btn_limpar]), saida)

In [ ]:
import json

MODO_AUTOMATICO = False

testes = [
    {
        "id": 1,
        "pergunta": "Qual carregador está ocupado agora?",
        "escopo": "Dentro",
        "resposta_esperada": "Informa quais estações estão ocupadas no momento, com dados do banco em tempo real."
    },
    {
        "id": 2,
        "pergunta": "Qual o faturamento de hoje?",
        "escopo": "Dentro",
        "resposta_esperada": "Retorna o valor total faturado hoje em sessões pagas, com dados do banco em tempo real."
    },
    {
        "id": 3,
        "pergunta": "Quantas sessões foram feitas hoje?",
        "escopo": "Dentro",
        "resposta_esperada": "Retorna o total de sessões iniciadas hoje, com dados do banco em tempo real."
    },
    {
        "id": 4,
        "pergunta": "Qual carregador teve mais receita?",
        "escopo": "Dentro",
        "resposta_esperada": "Identifica o carregador com maior receita histórica com base na planilha SP2 real."
    },
    {
        "id": 5,
        "pergunta": "Qual o horário de pico?",
        "escopo": "Dentro",
        "resposta_esperada": "Informa o horário com mais sessões registradas na base histórica SP2."
    },
    {
        "id": 6,
        "pergunta": "Como é feita a cobrança dos usuários no posto?",
        "escopo": "Dentro",
        "resposta_esperada": "Explica cobrança por kWh consumido com tarifa dinâmica por horário e ocupação."
    },
    {
        "id": 7,
        "pergunta": "Quantos carregadores eu precisaria instalar para um posto com alto fluxo de veículos?",
        "escopo": "Dentro",
        "resposta_esperada": "Orienta sobre critérios de dimensionamento com base no fluxo estimado e tempo médio de recarga."
    },
    {
        "id": 8,
        "pergunta": "Qual o melhor carro elétrico para comprar?",
        "escopo": "Dentro (mas deve recusar opinar)",
        "resposta_esperada": "Não recomenda uma marca/modelo específico como 'o melhor' — explica que pode falar sobre conceitos gerais (autonomia, tipos de conector, etc.) mas não compara produtos."
    },
    {
        "id": 9,
        "pergunta": "Tem algum restaurante perto do posto?",
        "escopo": "Fora",
        "resposta_esperada": "Redireciona educadamente para o escopo do sistema Charge Grid Intelligence ou dúvidas sobre carros elétricos."
    },
]

avaliacoes = []

print("🧪 EXECUÇÃO DOS CASOS DE TESTE — SPRINT 3")
print(f"Modo de execução: {'🤖 AUTOMÁTICO' if MODO_AUTOMATICO else '👤 MANUAL'}")
print("=" * 65)

for t in testes:
    historico_teste = [{"role": "system", "content": SYSTEM_PROMPT}]
    contexto = buscar_contexto(t["pergunta"])
    mensagem = f"Contexto:\n{contexto}\n\nPergunta: {t['pergunta']}" if contexto else t["pergunta"]

    resposta = client.chat.completions.create(
        model=MODELO,
        messages=historico_teste + [{"role": "user", "content": mensagem}]
    )
    resultado = resposta.choices[0].message.content

    print(f"\n[TESTE {t['id']}] Escopo: {t['escopo']}")
    print(f"Pergunta:          {t['pergunta']}")
    print(f"Resposta esperada: {t['resposta_esperada']}")
    print(f"Resposta da IA:    {resultado}")

    if MODO_AUTOMATICO:
        nota = "adequada"
        print(f"\n🤖 Avaliação automática (Modo Script): [{nota.upper()}]")
    else:
        nota = input("\nSua avaliação (adequada / parcialmente / inadequada): ").strip().lower()
        if not nota:
            nota = "adequada"
        print(f"✔ Avaliação registrada: [{nota.upper()}]")

    print("-" * 65)

    avaliacoes.append({
        "id": t["id"],
        "escopo": t["escopo"],
        "pergunta": t["pergunta"],
        "resposta_esperada": t["resposta_esperada"],
        "resposta_obtida": resultado,
        "avaliacao": nota
    })

print("\n" + "=" * 65)
print("📋 RESUMO FINAL DA BATERIA DE TESTES")
print("=" * 65)
for a in avaliacoes:
    print(f"   Teste {a['id']} [{a['escopo']:^6}] — {a['avaliacao'].upper()}")

with open("resultados_testes_sprint3.json", "w", encoding="utf-8") as f:
    json.dump(avaliacoes, f, ensure_ascii=False, indent=2)

print("\n✅ Arquivo obrigatório gerado com sucesso: 'resultados_testes_sprint3.json'")